In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/06_genai/05_sql_executor.py

# Phase 12 — Answer Generator

This notebook converts validated SQL results into grounded
natural-language business answers.

Architecture:

User Question
↓
SQL Result
↓
Result Validation
↓
Answer Generation
↓
Business Explanation

Important principle:

The answer generator must use actual query results.

It must NOT invent, estimate, or modify numerical values.


In [0]:
import json
import math
from datetime import date, datetime

print("Answer Generator initialized.")

In [0]:
def serialize_value(value):
    """
    Convert Spark/Python values into JSON-safe values.
    """

    if value is None:
        return None

    if isinstance(value, (datetime, date)):
        return value.isoformat()

    if isinstance(value, float):

        if math.isnan(value) or math.isinf(value):
            return None

        return round(value, 6)

    return value

In [0]:
def dataframe_to_records(
    dataframe,
    max_rows=100
):
    """
    Convert Spark DataFrame results into
    JSON-compatible records.

    max_rows prevents unnecessarily large
    result payloads from being sent downstream.
    """

    if dataframe is None:
        return []

    rows = dataframe.limit(max_rows).collect()

    records = []

    for row in rows:

        record = {}

        for column in dataframe.columns:

            record[column] = serialize_value(
                row[column]
            )

        records.append(record)

    return records

In [0]:
def validate_query_result(result):
    """
    Validate the output returned by execute_sql().
    """

    if not isinstance(result, dict):

        return {
            "valid": False,
            "reason": "Execution result is not a dictionary."
        }

    if not result.get("success", False):

        return {
            "valid": False,
            "reason": result.get(
                "error",
                "SQL execution failed."
            )
        }

    dataframe = result.get("dataframe")

    if dataframe is None:

        return {
            "valid": False,
            "reason": "SQL execution returned no DataFrame."
        }

    row_count = result.get(
        "row_count",
        0
    )

    if row_count == 0:

        return {
            "valid": True,
            "empty": True,
            "reason": "No matching records were found."
        }

    return {
        "valid": True,
        "empty": False,
        "reason": "Query result is valid."
    }

In [0]:
def build_answer_context(
    question,
    sql,
    result,
    source_table
):
    """
    Build a structured context object for answer generation.
    """

    validation = validate_query_result(result)

    if not validation["valid"]:

        return {
            "question": question,
            "sql": sql,
            "source_table": source_table,
            "result_status": "ERROR",
            "result": [],
            "row_count": 0,
            "error": validation["reason"]
        }

    if validation.get("empty"):

        return {
            "question": question,
            "sql": sql,
            "source_table": source_table,
            "result_status": "NO_DATA",
            "result": [],
            "row_count": 0,
            "error": None
        }

    records = dataframe_to_records(
        result["dataframe"],
        max_rows=100
    )

    return {
        "question": question,
        "sql": sql,
        "source_table": source_table,
        "result_status": "SUCCESS",
        "result": records,
        "row_count": result["row_count"],
        "error": None
    }

In [0]:
def generate_deterministic_answer(
    context
):
    """
    Generate a basic factual answer directly
    from the query result.

    This provides a safe fallback if an LLM
    is unavailable.
    """

    status = context["result_status"]

    if status == "ERROR":

        return (
            "I could not answer the question because "
            "the data query failed. "
            f"Reason: {context['error']}"
        )

    if status == "NO_DATA":

        return (
            "No matching records were found "
            "for this question."
        )

    records = context["result"]

    if not records:

        return "No data was returned."

    question = context["question"]

    # ---------------------------------------------
    # Single-value result
    # ---------------------------------------------

    if len(records) == 1:

        record = records[0]

        if len(record) == 1:

            key = list(record.keys())[0]
            value = record[key]

            return (
                f"The result for your question "
                f"is {value}."
            )

    # ---------------------------------------------
    # Ranking / comparison result
    # ---------------------------------------------

    first_record = records[0]

    if len(first_record) >= 2:

        keys = list(first_record.keys())

        dimension = keys[0]
        metric = keys[1]

        first_dimension = first_record[dimension]
        first_metric = first_record[metric]

        return (
            f"Based on the available data, "
            f"{first_dimension} has the highest "
            f"{metric.replace('_', ' ')} at "
            f"{first_metric}."
        )

    # ---------------------------------------------
    # Generic fallback
    # ---------------------------------------------

    return (
        f"The query returned "
        f"{len(records)} result rows."
    )

In [0]:
answer = generate_deterministic_answer(
    context
)

print("ANSWER")
print("=" * 60)
print(answer)

In [0]:
question = "Which region generated the highest revenue?"

sql = """
SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
GROUP BY region
ORDER BY total_revenue DESC
LIMIT 10
"""

In [0]:
context = build_answer_context(
    question=question,
    sql=sql,
    result=result,
    source_table="genai_copilot.gold.region_sales"
)

print(
    json.dumps(
        context,
        indent=2,
        default=str
    )
)

In [0]:
answer = generate_deterministic_answer(context)

print(answer)

In [0]:
def build_response(
    question,
    result,
    source_table,
    answer
):
    """
    Build the standard response format
    used by the application.
    """

    return {
        "question": question,

        "answer": answer,

        "data": dataframe_to_records(
            result.get("dataframe"),
            max_rows=100
        ) if result.get("success") else [],

        "sql": result.get(
            "sql"
        ),

        "source": source_table,

        "validation": result.get(
            "validation"
        ),

        "row_count": result.get(
            "row_count",
            0
        ),

        "execution_time_ms": result.get(
            "execution_time_ms"
        ),

        "success": result.get(
            "success",
            False
        ),

        "error": result.get(
            "error"
        )
    }

In [0]:
response = build_response(
    question=question,
    result=result,
    source_table="genai_copilot.gold.region_sales",
    answer=answer
)

print(
    json.dumps(
        response,
        indent=2,
        default=str
    )
)

##12.13 Why aren't we using the LLM yet?

This is intentional.

Our architecture should be:

                 ┌──────────────────────┐
                 │       User           │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │ Question Classifier  │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │    SQL Generator     │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │    SQL Validator     │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │    SQL Executor      │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │ Actual Spark Result  │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │ Answer Generator     │
                 └──────────┬───────────┘
                            ↓
                 ┌──────────────────────┐
                 │      UI Answer       │
                 └──────────────────────┘

The deterministic layer gives us a safe fallback.

Later, we'll add:

Actual Result
      ↓
Structured Context
      ↓
LLM
      ↓
Natural Language Explanation

The LLM will never be responsible for obtaining the numbers.

12.14 Test "No Data"

Let's verify error handling.

Run:

In [0]:

empty_sql = """
SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
WHERE region = 'Atlantis'
GROUP BY region
"""

empty_result = execute_sql(
    empty_sql
)

empty_context = build_answer_context(
    question="What is the revenue in Atlantis?",
    sql=empty_sql,
    result=empty_result,
    source_table="genai_copilot.gold.region_sales"
)

empty_answer = generate_deterministic_answer(
    empty_context
)

print(empty_answer)

In [0]:
bad_result = execute_sql(
    """
    DROP TABLE genai_copilot.gold.region_sales
    """
)

bad_context = build_answer_context(
    question="Delete the region sales table",
    sql="DROP TABLE genai_copilot.gold.region_sales",
    result=bad_result,
    source_table="genai_copilot.gold.region_sales"
)

bad_answer = generate_deterministic_answer(
    bad_context
)

print(bad_answer)

This demonstrates:

LLM-generated SQL → security layer → rejected → no execution → friendly answer.

12.16 Phase 12 checkpoint

At this point our project has:

Component	Status
Schema Profiler	✅
Question Classifier	✅
SQL Generator	✅
SQL Validator	✅
SQL Executor	✅
Result Validation	✅
Deterministic Answer Generator	✅
Structured Response	✅
No-data handling	✅
Error handling	✅